Imagine a store where reps had the following total revenues:
\\$100, \\$100, \\$90, \\$80. (Notice the tie for first place).

    row_number(): Will literally just count rows. It will arbitrarily break the tie.
        Result: 1, 2, 3, 4.

    rank(): Will assign the same rank to ties, but will leave a gap in the numbers afterward.
        Result: 1, 1, 3, 4. (Notice there is no rank 2. The person who scored $90 is considered 3rd place).

    dense_rank(): Will assign the same rank to ties, but will NOT leave a gap.
        Result: 1, 1, 2, 3. (The person who scored $90 is considered 2nd place).

Generally, business users prefer dense_rank() for "Top N" reports so they don't see missing numbers in their dashboards!

### Top 3 highest-grossing sales reps for each store, based on their total sales revenue.

In [1]:
from pyspark.sql import SparkSession
import getpass

username = getpass.getuser()

In [2]:
spark = SparkSession.builder \
.config("spark.port.ui", 0) \
.config("spark.sql.warehouse.dir", f"/user/{username}/warehouse") \
.enableHiveSupport() \
.master("yarn") \
.appName("dense_rank_025320") \
.getOrCreate()

In [3]:
from pyspark.sql import Window
from pyspark.sql import functions as F

sales_df = spark.range(100000) \
    .withColumn("store_id", (F.rand() * 50).cast("int")) \
    .withColumn("sales_rep_id", (F.rand() * 100).cast("int")) \
    .withColumn("transaction_revenue", (F.rand() * 450).cast("double"))

In [4]:
rep_totals_df = sales_df.groupBy("store_id", "sales_rep_id").agg(F.sum("transaction_revenue").alias("total_revenue"))

In [5]:
window_spec = Window.partitionBy("store_id").orderBy("total_revenue")

In [6]:
rank_agg = rep_totals_df.withColumn("rep_rank", (F.dense_rank().over(window_spec)))

In [7]:
top_3_reps_df = rank_agg.filter(F.col("rep_rank") <= 3)

In [8]:
top_3_reps_df.orderBy("store_id", "rep_rank").show()

+--------+------------+------------------+--------+
|store_id|sales_rep_id|     total_revenue|rep_rank|
+--------+------------+------------------+--------+
|       0|          85|1754.4534447138233|       1|
|       0|           9|2505.3008885453737|       2|
|       0|          22|2507.6308747414723|       3|
|       1|          11| 2363.204287319752|       1|
|       1|          14| 2375.013385967106|       2|
|       1|          25|2557.2352194018704|       3|
|       2|          10|  1895.45062746511|       1|
|       2|          25|2899.0271769399756|       2|
|       2|          34|3020.1539451373355|       3|
|       3|           6| 1241.153742260963|       1|
|       3|          82|1809.7522327248337|       2|
|       3|          40| 2327.215518569272|       3|
|       4|          81|1305.5531854105325|       1|
|       4|          44| 1884.314033371361|       2|
|       4|          22| 2221.132858431874|       3|
|       5|           8|1714.3087575939505|       1|
|       5|  